# Lab 02 — Azure Services & Shared Lakehouse Setup
## Parameterized Bronze ingestion with Auto Loader

This notebook rebuilds the Databricks-side part of Lab 2 from a clean state.

It covers:
- Bronze, Silver, and Gold schemas backed by external ADLS storage
- parameterized ingestion paths and target names
- explicit source schema
- Auto Loader ingestion
- metadata columns
- checkpoint-based idempotency
- Delta write to the Bronze schema
- optional validation and idempotency checks
- Databricks Job parameter guidance
- legacy Service Principal access through the supervisor-recommended direct `abfss://` method

> `dbutils.fs.mount()` is not used because the academy shared-access cluster does not whitelist it.


# Lab task mapping

| # | Task | Status / coverage |
|---|---|---|
| 1 | Create personal ADLS container `parvinbadalov` | Completed outside notebook |
| 2 | Create UC external location `parvinbadalov_external_location` | Completed outside notebook |
| 3 | Test external location read/write access | Completed outside notebook |
| 4 | Create Bronze, Silver, Gold schemas backed by external storage | Created in this notebook |
| 5 | Create personal Azure Key Vault-backed secret scope | Completed outside notebook |
| 6 | Create source/ingestion folder | Completed outside notebook / validated by ingestion |
| 7 | Upload source files | Completed outside notebook |
| 8 | Build Bronze ingestion notebook | This notebook |
| 9 | Add metadata columns | Yes |
| 10 | Make ingestion idempotent | Yes |
| 11 | Write data as Delta table to Bronze schema | Yes |
| 12 | Create Databricks Job | Configure in Jobs UI |
| 13 | Configure Job to use shared all-purpose cluster | Use `GP1` |
| 14 | Run and validate the Job | Validation controls included |
| 15 | Configure legacy Service Principal access from secret scope | Yes |
| 16 | Create legacy `dbutils.fs.mount` | Not supported on current shared cluster |
| 17 | Test legacy storage access | Yes, via direct `abfss://` access |
| 18 | Final validation and screenshots/documentation | Checklist included at the end |


# 1. Parameters

The notebook uses Databricks widgets so the same code can run:

- interactively with default values;
- as a Databricks Job with task parameters;
- in another environment by changing parameter values only.

For normal Job runs, keep the optional validation switches set to `false`.


In [0]:
# Core Azure storage parameters
dbutils.widgets.text("storage_account", "dlspl21databricks")
dbutils.widgets.text("container", "parvinbadalov")

# Unity Catalog names
dbutils.widgets.text("catalog", "dbr_dev")
dbutils.widgets.text("bronze_schema", "parvinbadalov_bronze")
dbutils.widgets.text("silver_schema", "parvinbadalov_silver")
dbutils.widgets.text("gold_schema", "parvinbadalov_gold")
dbutils.widgets.text("target_table", "orders_autoloader")

# ADLS folder names
dbutils.widgets.text("source_dir", "ingestion")
dbutils.widgets.text("bronze_dir", "bronze")
dbutils.widgets.text("silver_dir", "silver")
dbutils.widgets.text("gold_dir", "gold")
dbutils.widgets.text("checkpoint_dir", "checkpoints/orders_autoloader")

# Shared secret scope for legacy SPN access
dbutils.widgets.text("secret_scope", "default2")

# Optional development/testing switches
dbutils.widgets.dropdown("run_validation", "true", ["true", "false"])
dbutils.widgets.dropdown("run_idempotency_test", "false", ["true", "false"])
dbutils.widgets.dropdown("run_spn_access_test", "false", ["true", "false"])


# 2. Build paths and object names

Full ABFSS paths and fully qualified Unity Catalog names are derived from the parameters above.


In [0]:
storage_account = dbutils.widgets.get("storage_account")
container = dbutils.widgets.get("container")

catalog = dbutils.widgets.get("catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
silver_schema = dbutils.widgets.get("silver_schema")
gold_schema = dbutils.widgets.get("gold_schema")
target_table = dbutils.widgets.get("target_table")

source_dir = dbutils.widgets.get("source_dir").strip("/")
bronze_dir = dbutils.widgets.get("bronze_dir").strip("/")
silver_dir = dbutils.widgets.get("silver_dir").strip("/")
gold_dir = dbutils.widgets.get("gold_dir").strip("/")
checkpoint_dir = dbutils.widgets.get("checkpoint_dir").strip("/")

SECRET_SCOPE = dbutils.widgets.get("secret_scope")

run_validation = dbutils.widgets.get("run_validation").lower() == "true"
run_idempotency_test = dbutils.widgets.get("run_idempotency_test").lower() == "true"
run_spn_access_test = dbutils.widgets.get("run_spn_access_test").lower() == "true"

storage_root = f"abfss://{container}@{storage_account}.dfs.core.windows.net"

source_path = f"{storage_root}/{source_dir}/"
bronze_location = f"{storage_root}/{bronze_dir}/"
silver_location = f"{storage_root}/{silver_dir}/"
gold_location = f"{storage_root}/{gold_dir}/"
checkpoint_path = f"{storage_root}/{checkpoint_dir}/"

bronze_table = f"{catalog}.{bronze_schema}.{target_table}"

print("Source path:", source_path)
print("Bronze schema location:", bronze_location)
print("Silver schema location:", silver_location)
print("Gold schema location:", gold_location)
print("Checkpoint path:", checkpoint_path)
print("Target Bronze table:", bronze_table)


# 3. Create externally backed Bronze, Silver, and Gold schemas

Each schema is created with its own `MANAGED LOCATION` under the personal ADLS container.

Only the Bronze schema is used for ingestion in this lab.  
Silver and Gold are created now for the medallion architecture setup required by Task 4.


In [0]:
spark.sql(f'''
CREATE SCHEMA IF NOT EXISTS {catalog}.{bronze_schema}
MANAGED LOCATION '{bronze_location}'
''')

spark.sql(f'''
CREATE SCHEMA IF NOT EXISTS {catalog}.{silver_schema}
MANAGED LOCATION '{silver_location}'
''')

spark.sql(f'''
CREATE SCHEMA IF NOT EXISTS {catalog}.{gold_schema}
MANAGED LOCATION '{gold_location}'
''')


In [0]:
# Validate that all three schemas exist
display(
    spark.sql(
        f"SHOW SCHEMAS IN {catalog} LIKE 'parvinbadalov*'"
    )
)


# 4. Define the explicit source schema

The source CSV contains:

- `Order_ID` → string
- `Product` → string
- `Category` → string
- `Quantity` → integer
- `Price` → double
- `City` → string
- `Date` → date


In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DoubleType,
    DateType
)

source_schema = StructType([
    StructField("Order_ID", StringType(), True),
    StructField("Product", StringType(), True),
    StructField("Category", StringType(), True),
    StructField("Quantity", IntegerType(), True),
    StructField("Price", DoubleType(), True),
    StructField("City", StringType(), True),
    StructField("Date", DateType(), True)
])


# 5. Read incrementally with Auto Loader

An explicit schema is supplied, so a separate `cloudFiles.schemaLocation` is not required for schema inference.

The checkpoint is still required because it stores Auto Loader progress and prevents already processed files from being loaded again.


In [0]:
from pyspark.sql.functions import current_timestamp, current_date, col

df_bronze = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("dateFormat", "yyyy-MM-dd")
    .schema(source_schema)
    .load(source_path)
)


# 6. Add required metadata columns

The lab requires:

- source filename/path;
- ingestion timestamp;
- load date.


In [0]:
df_bronze = (
    df_bronze
    .withColumn("_source_file", col("_metadata.file_path"))
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_load_date", current_date())
)


# 7. Write the Bronze Delta table

`availableNow=True` processes all currently available new files and then stops.

This makes the notebook suitable for:
- manual runs;
- scheduled Jobs;
- file-arrival-triggered Jobs.

The Job trigger decides **when the notebook starts**.  
`availableNow=True` decides **how the streaming query processes available files once the notebook is running**.


In [0]:
(
    df_bronze.writeStream
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(bronze_table)
)


# 8. Optional validation

For interactive development, set:

- `run_validation = true`

For normal Job runs, set:

- `run_validation = false`


In [0]:
if run_validation:
    display(spark.table(bronze_table))

    row_count = spark.table(bronze_table).count()
    print("Bronze row count:", row_count)

    required_metadata_columns = {
        "_source_file",
        "_ingested_at",
        "_load_date"
    }

    actual_columns = set(spark.table(bronze_table).columns)
    missing_columns = required_metadata_columns - actual_columns

    assert not missing_columns, f"Missing metadata columns: {missing_columns}"

    print("Metadata validation passed.")


# 9. Optional idempotency test

For manual testing, set:

- `run_idempotency_test = true`

With no new files, the row count must stay unchanged.

For a regular production-style Job run, keep this `false` so the notebook does not intentionally run the ingestion twice.


In [0]:
if run_idempotency_test:
    count_before = spark.table(bronze_table).count()

    (
        df_bronze.writeStream
        .option("checkpointLocation", checkpoint_path)
        .trigger(availableNow=True)
        .toTable(bronze_table)
    )

    count_after = spark.table(bronze_table).count()

    print("Before rerun:", count_before)
    print("After rerun :", count_after)

    assert count_before == count_after, "Idempotency test failed"

    print("Idempotency test passed.")


# 10. Databricks Job configuration

Create or update the Job to point to this notebook.

## Recommended task parameters

| Key | Value |
|---|---|
| `storage_account` | `dlspl21databricks` |
| `container` | `parvinbadalov` |
| `catalog` | `dbr_dev` |
| `bronze_schema` | `parvinbadalov_bronze` |
| `silver_schema` | `parvinbadalov_silver` |
| `gold_schema` | `parvinbadalov_gold` |
| `target_table` | `orders_autoloader` |
| `source_dir` | `ingestion` |
| `bronze_dir` | `bronze` |
| `silver_dir` | `silver` |
| `gold_dir` | `gold` |
| `checkpoint_dir` | `checkpoints/orders_autoloader` |
| `secret_scope` | `default2` |
| `run_validation` | `false` |
| `run_idempotency_test` | `false` |
| `run_spn_access_test` | `false` |

## Compute

Use the shared all-purpose cluster:

`GP1`

## Trigger recommendation

For this lab, a **file-arrival trigger** is a strong demonstration because the six CSV files can be uploaded one by one.

Suggested flow:

1. Upload January file and run once manually.
2. Configure file-arrival trigger on the `ingestion/` path.
3. Upload February through June one by one.
4. Confirm each new file causes the Job to run.
5. Confirm Auto Loader processes only files that were not already checkpointed.

A schedule is also valid, but file arrival is a better fit for this file-ingestion scenario.


# 11. Legacy Service Principal access from the shared secret scope

The shared scope contains:

- `tenant-id`
- `sp-databricks-adls-appid`
- `sp-databricks-adls-appkey`

The secret values must never be printed.


In [0]:
if run_spn_access_test:
    tenant_id = dbutils.secrets.get(
        scope=SECRET_SCOPE,
        key="tenant-id"
    )

    client_id = dbutils.secrets.get(
        scope=SECRET_SCOPE,
        key="sp-databricks-adls-appid"
    )

    client_secret = dbutils.secrets.get(
        scope=SECRET_SCOPE,
        key="sp-databricks-adls-appkey"
    )

    print("Client ID loaded:", bool(client_id))
    print("Client secret loaded:", bool(client_secret))
    print("Tenant ID loaded:", bool(tenant_id))


# 12. Supervisor-recommended legacy access alternative

The academy Shared access mode cluster blocks `dbutils.fs.mount()`.

Instead, when `run_spn_access_test = true`, configure Spark with Service Principal OAuth settings and validate direct `abfss://` access.


In [0]:
if run_spn_access_test:
    spark.conf.set(
        f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net",
        "OAuth"
    )

    spark.conf.set(
        f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net",
        "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
    )

    spark.conf.set(
        f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net",
        client_id
    )

    spark.conf.set(
        f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net",
        client_secret
    )

    spark.conf.set(
        f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net",
        f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
    )

    display(dbutils.fs.ls(f"{storage_root}/"))


# 13. Final validation and screenshot checklist

Capture screenshots of:

1. Personal ADLS container.
2. Ingestion folder with source CSV files.
3. Unity Catalog external location.
4. Bronze, Silver, and Gold schemas.
5. Personal Azure Key Vault-backed secret scope.
6. Bronze table with:
   - `_source_file`
   - `_ingested_at`
   - `_load_date`
7. Successful idempotency test:
   - `Before rerun: X`
   - `After rerun: X`
8. Successful Databricks Job run.
9. Job configured on shared all-purpose cluster `GP1`.
10. File-arrival trigger configuration, if used.
11. Shared SPN credentials loaded successfully without exposing values.
12. Successful direct `abfss://` storage listing with Service Principal OAuth.

## Legacy mount note

`dbutils.fs.mount()` was not used because it is not whitelisted on the academy Shared access mode cluster.

The supervisor-recommended replacement was used instead:
- read SPN credentials from the shared secret scope;
- configure Spark OAuth settings;
- access ADLS directly through `abfss://`.
